In [1]:
import warnings
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

from dpec.forecasting.backtest import score, walk_forward
from dpec.forecasting.model import FORMULAS, NEEDS, fit, shared_model, with_form
from dpec.forecasting.targets import forecast_table

warnings.filterwarnings("ignore")  # statsmodels talks a lot; nothing it says changes these results

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
table = forecast_table(ROOT / "data/warehouse.duckdb")
pred = pd.read_parquet(ROOT / "data/forecasts/backtest.parquet")
print(table.shape, pred.shape)


(1846, 17) (18012, 11)


In [2]:
last = table["weekend"].max()
history, today = table[table["weekend"] < last], table[table["weekend"] == last]
h, t = with_form(history, today)
for checkpoint in FORMULAS:
    m = fit(h, checkpoint)
    print(checkpoint, m.params.round(3).to_dict(), "R2", round(m.rsquared, 2), "rows", int(m.nobs))

after_practice {'Intercept': 0.091, 'team_form': 0.966, 'driver_vs_team': 0.828} R2 0.65 rows 1805
after_quali {'Intercept': 0.186, 'quali_gap_pct': 0.6, 'team_form': 0.381, 'driver_vs_team': 0.359} R2 0.76 rows 1692


In [3]:
def with_track(history, today):
    """Every race counts the same, plus one random effect per circuit (MixedLM)."""
    h, t = with_form(history, today)
    guess = t["team_form"].copy()
    for checkpoint, needs in NEEDS.items():
        if any(column not in t for column in needs):
            continue
        train = h.dropna(subset=needs + ["team_form"])
        m = smf.mixedlm(FORMULAS[checkpoint], train, groups=train["circuit"]).fit()
        circuit = {name: effect.iloc[0] for name, effect in m.random_effects.items()}
        known = t.dropna(subset=needs).index
        guess[known] = m.predict(t.loc[known]) + t.loc[known, "circuit"].map(circuit).fillna(0.0)
    return guess

equal_weights = partial(shared_model, halflife=None)
track = walk_forward(table, {"equal_weights": equal_weights, "with_track": with_track})
print(score(track).to_string(index=False, float_format="%.3f"))

    checkpoint    forecaster  weekends  rows  mae_pct  mae_s  median_rho
after_practice equal_weights        78  1501    0.553  0.498       0.774
after_practice    with_track        78  1501    0.569  0.512       0.771
   after_quali equal_weights        78  1501    0.513  0.462       0.831
   after_quali    with_track        78  1501    0.535  0.481       0.831


In [4]:
def with_era(history, today):
    """Every race counts the same, plus the rules era: from 2026 on, every coefficient may change."""
    h, t = with_form(history, today)
    h = h.assign(era_2026=(h["season"] >= 2026).astype(int))
    t = t.assign(era_2026=(t["season"] >= 2026).astype(int))
    guess = t["team_form"].copy()
    for checkpoint, needs in NEEDS.items():
        if any(column not in t for column in needs):
            continue
        formula = FORMULAS[checkpoint]
        if h["era_2026"].any():  # only once a 2026 race is in the history
            formula = formula.replace("~ ", "~ (") + ") * era_2026"
        train = h.dropna(subset=needs + ["team_form"])
        known = t.dropna(subset=needs).index
        guess[known] = smf.ols(formula, train).fit().predict(t.loc[known])
    return guess

forecasters = {"equal_weights": equal_weights, "with_era": with_era, "shared_model": shared_model}
era = walk_forward(table, forecasters, first_season=2026)
simple = pred[(pred["season"] == 2026) & pred["forecaster"].isin(["team_form", "per_year_model"])]
both = pd.concat([era, simple])
both["races"] = np.where(both["round"] <= 5, "1-5", "6+")
both["abs_error"] = (both["predicted"] - both["actual"]).abs()
print(both.pivot_table(index=["checkpoint", "races"], columns="forecaster", values="abs_error").round(3).to_string())


forecaster            equal_weights  per_year_model  shared_model  team_form  with_era
checkpoint     races                                                                  
after_practice 1-5            0.844           1.021         0.838      0.803     1.034
               6+             0.694           0.976         0.713      0.755     0.761
after_quali    1-5            0.721           0.570         0.626      0.803     0.682
               6+             0.595           0.620         0.571      0.755     0.600


In [5]:
tune_table = table[table["season"] <= 2024]
candidates = {f"halflife_{h}": partial(shared_model, halflife=h) for h in [6, 12, 24, 48]}
candidates["equal_weights"] = equal_weights
tune = walk_forward(tune_table, candidates)
print(score(tune).to_string(index=False, float_format="%.3f"))

    checkpoint    forecaster  weekends  rows  mae_pct  mae_s  median_rho
after_practice   halflife_24        41   775    0.470  0.423       0.771
after_practice   halflife_48        41   775    0.470  0.423       0.771
after_practice equal_weights        41   775    0.470  0.423       0.765
after_practice   halflife_12        41   775    0.470  0.423       0.771
after_practice    halflife_6        41   775    0.473  0.426       0.773
   after_quali    halflife_6        41   775    0.450  0.405       0.847
   after_quali   halflife_12        41   775    0.451  0.406       0.833
   after_quali   halflife_24        41   775    0.452  0.407       0.842
   after_quali   halflife_48        41   775    0.453  0.408       0.839
   after_quali equal_weights        41   775    0.454  0.409       0.830


**Conclusions:**

1. The target is relative to the best car of that race, so the track's general speed (Monaco slow, Monza fast) is already gone. What's left is "this car suits this track", which is real but small, and the model sees each circuit only 3 to 4 times in history: an offset learned from 3 noisy races mostly fits noise. Rejected; it may come back later as a car-and-track interaction in the car root-cause module. Your plan said "mixed effects → Bayesian": the first step didn't help, so a Bayesian version isn't worth building now (LATER, if ever).
2. With the switch, the 2026 coefficients are learned only from 2026 rows. After 1 to 5 races that's 20 to 100 rows: far too few, so it guesses wildly. The recency weights do the same job gently: 2026 races weigh most, but older races still steady the fit. The honest summary: after a big rules change, expect about 5 races where simple guesses are as good or better; the model catches up after that.
3. 12: never more than 0.001 from the best at either checkpoint, and about half a season, which fits "learn during the season". The choice barely matters, which is itself a result: write it down and move on.

---